In [1]:
import numpy as np
import cv2
import viser
import torch
import torch.nn.functional as F
import time
import pypose as pp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
server = viser.ViserServer()
server.scene.add_grid(
    "/grid",
    width=10.0,
    height=10.0,
    position=np.array([0.0, 0.0, 0.0]),
    cell_color=(127, 127, 127),
    section_color=(127, 127, 127),
    infinite_grid=False,

)

╭──────────────── viser ────────────────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

GridHandle(width=10.0, height=10.0, plane='xy', cell_color=(127, 127, 127), cell_thickness=1.0, cell_size=0.5, section_color=(127, 127, 127), section_thickness=1.0, section_size=1.0, infinite_grid=False, fade_distance=100.0, fade_strength=1.0, fade_from='camera', shadow_opacity=0.125)

(viser) Connection opened (0, 1 total), 7 persistent messages

In [3]:
# datafile = np.load("../output/peter_1_03.npy", allow_pickle=True)
# datafile = np.load("../output/olympics_2.npy", allow_pickle=True)
datafile = np.load("../output/peter_3_01.npy", allow_pickle=True)

In [4]:
datafile[0].keys()

dict_keys(['bbox', 'focal_length', 'pred_keypoints_3d', 'pred_keypoints_2d', 'pred_vertices', 'pred_cam_t', 'pred_pose_raw', 'global_rot', 'body_pose_params', 'hand_pose_params', 'scale_params', 'shape_params', 'expr_params', 'mask', 'pred_joint_coords', 'pred_global_rots', 'mhr_model_params', 'contact_logits', 'contact_probs', 'lhand_bbox', 'rhand_bbox'])

In [5]:
datafile[0]["pred_joint_coords"].shape

(127, 3)

In [6]:
def gaussian_smooth_1d(tensor, kernel_size=5, sigma=1.0):
    x = torch.arange(kernel_size, dtype=tensor.dtype, device=tensor.device) - (kernel_size - 1) / 2
    kernel = torch.exp(-x**2 / (2 * sigma**2))
    kernel = kernel / kernel.sum()
    kernel = kernel.view(1, 1, -1)
    
    tensor_reshaped = tensor.T.unsqueeze(1)
    smoothed = F.conv1d(tensor_reshaped, kernel, padding=kernel_size // 2)
    return smoothed.squeeze(1).T

def ensure_quat_continuity(quats):
    # Flip quaternions to ensure shortest path (q and -q represent same rotation)
    for i in range(1, len(quats)):
        if torch.dot(quats[i], quats[i-1]) < 0:
            quats[i] = -quats[i]
    return quats

identity_coeffs = []
model_params = []
camera_ts = []

for frame_data in datafile:
    identity_coeffs.append(frame_data['shape_params'])
    model_params.append(frame_data['mhr_model_params'])

    camera_t = frame_data["pred_cam_t"].copy()
    camera_t[[1, 2]] *= -1
    # model_params[-1][:3] = camera_t * 1
    camera_ts.append(camera_t)

identity_coeffs = torch.tensor(identity_coeffs).to(device)
identity_coeffs = identity_coeffs.mean(dim=0).unsqueeze(0)

model_params = torch.tensor(model_params).to(device)
model_params[:, 142:] = model_params[:, 142:].mean(dim=0)

model_params[:, :3] = gaussian_smooth_1d(model_params[:, :3], kernel_size=7, sigma=2.0)
model_params[:, 6:142] = gaussian_smooth_1d(model_params[:, 6:142], kernel_size=5, sigma=1.0)

camera_ts = torch.tensor(camera_ts).to(device)
camera_ts = gaussian_smooth_1d(camera_ts, kernel_size=7, sigma=2.0)
camera_ts = camera_ts.cpu().numpy()
zero_cam = camera_ts.mean(axis=0)

global_rot = pp.euler2SO3(model_params[:, 3:6])
global_rot = ensure_quat_continuity(global_rot)
global_rot = gaussian_smooth_1d(global_rot, kernel_size=5, sigma=1.0)
model_params[:, 3:6] = pp.SO3(global_rot).euler()

face_expr_coeffs = torch.zeros(1, 72, device=device)     # Facial

datafile[0]['pred_pose_raw'].shape

/tmp/ipykernel_3953664/3061265822.py:31: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  identity_coeffs = torch.tensor(identity_coeffs).to(device)


(266,)

In [7]:
scripted_mhr_model = torch.jit.load("../../MHR/assets/mhr_model.pt").to(device)
faces = scripted_mhr_model.character_torch.mesh.faces.cpu().numpy()

In [8]:
# pretransfrom = pp.identity_SE3()
# pretransfrom[3:] = pp.euler2SO3([np.pi/2, 0, 0])

pre_rotation = pp.euler2SO3([np.pi/2, 0, 0])
pre_rotation = pre_rotation.matrix().numpy()
# pre_rotation = np.expand_dims(pre_rotation, axis=0)

# zero_cam = datafile[0]['pred_cam_t'].copy()
# zero_cam[[1, 2]] *= -1


In [9]:
time.sleep(4)

for i, frame_data in enumerate(datafile):

    vertices, skeleton_state = scripted_mhr_model(identity_coeffs, model_params[i:i+1], face_expr_coeffs)
    vertices = vertices[0].cpu().numpy() / 100

    # vertices = frame_data["pred_vertices"]

    # camera_t = frame_data["pred_cam_t"].copy()
    # camera_t[[1, 2]] *= -1

    vertices = vertices - zero_cam + camera_ts[i]
    vertices = (pre_rotation @ vertices.T).T


    server.add_mesh_simple(
        name="/mhr",
        vertices=vertices,
        faces=faces,
        color=[10, 70, 210],
    )

    time.sleep(1/30)

/tmp/ipykernel_3953664/2834566509.py:17: DeprecationWarning: ViserServer.add_mesh_simple has been deprecated, use ViserServer.scene.add_mesh_simple instead. Alternatively, pin to `viser<0.2.0`.
  server.add_mesh_simple(


In [8]:
joint_names = scripted_mhr_model.get_joint_names()
joint_names

['body_world',
 'root',
 'l_upleg',
 'l_lowleg',
 'l_foot',
 'l_talocrural',
 'l_subtalar',
 'l_transversetarsal',
 'l_ball',
 'l_lowleg_twist1_proc',
 'l_lowleg_twist2_proc',
 'l_lowleg_twist3_proc',
 'l_lowleg_twist4_proc',
 'l_upleg_twist0_proc',
 'l_upleg_twist1_proc',
 'l_upleg_twist2_proc',
 'l_upleg_twist3_proc',
 'l_upleg_twist4_proc',
 'r_upleg',
 'r_lowleg',
 'r_foot',
 'r_talocrural',
 'r_subtalar',
 'r_transversetarsal',
 'r_ball',
 'r_lowleg_twist1_proc',
 'r_lowleg_twist2_proc',
 'r_lowleg_twist3_proc',
 'r_lowleg_twist4_proc',
 'r_upleg_twist0_proc',
 'r_upleg_twist1_proc',
 'r_upleg_twist2_proc',
 'r_upleg_twist3_proc',
 'r_upleg_twist4_proc',
 'c_spine0',
 'c_spine1',
 'c_spine2',
 'c_spine3',
 'r_clavicle',
 'r_uparm',
 'r_lowarm',
 'r_wrist_twist',
 'r_wrist',
 'r_pinky0',
 'r_pinky1',
 'r_pinky2',
 'r_pinky3',
 'r_pinky_null',
 'r_ring1',
 'r_ring2',
 'r_ring3',
 'r_ring_null',
 'r_middle1',
 'r_middle2',
 'r_middle3',
 'r_middle_null',
 'r_index1',
 'r_index2',
 'r

In [12]:
indices = [joint_names.index(name) for name in [
    'root',
    'l_upleg',
    'l_lowleg',
    'l_foot',
    # 'l_talocrural',
    'l_subtalar',
    # 'l_transversetarsal',
    'l_ball',
    'r_upleg',
    'r_lowleg',
    'r_foot',
    # 'r_talocrural',
    'r_subtalar',
    # 'r_transversetarsal',
    'r_ball',
    'c_spine0',
    'c_spine1',
    'c_spine2',
    'c_spine3',
    'r_clavicle',
    'r_uparm',
    'r_lowarm',
    'r_wrist',
    'l_clavicle',
    'l_uparm',
    'l_lowarm',
    'l_wrist',
    'c_neck',
    'c_head',
    'r_eye',
    'l_eye',
]]
print(indices)
print(len(indices))

[1, 2, 3, 4, 6, 8, 18, 19, 20, 22, 24, 34, 35, 36, 37, 38, 39, 40, 42, 74, 75, 76, 78, 110, 113, 122, 124]
27


In [13]:
indices = [joint_names.index(name) for name in [
    'l_subtalar',
    'l_ball',
    'r_subtalar',
    'r_ball',
    'r_wrist',
    'l_wrist',
]]
print(indices)
print(len(indices))

[6, 8, 22, 24, 42, 78]
6
